# **Model Benchmarking**
---

This notebook systematically measure inference latency, throughput (FPS), and peak memory usage for each of the three YOLOv8 variants (nano, small, and medium) under identical conditions.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from detection_system.loader import load_model
from detection_system.benchmark import benchmark_model, compare_benchmarks, BenchmarkResult
from detection_system.utils import load_config, save_figure, save_table, frames_from_video

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
CONF = cfg["model"]["confidence_threshold"]
N_WARMUP = cfg["benchmark"]["n_warmup_frames"]
N_BENCH = cfg["benchmark"]["n_benchmark_frames"]
VARIANTS = cfg["benchmark"]["variants_to_compare"]

print(f"Variants to benchmark : {VARIANTS}")
print(f"Warm-up frames        : {N_WARMUP}")
print(f"Benchmark frames      : {N_BENCH}")
print(f"Confidence threshold  : {CONF}")
print("Setup complete")

Variants to benchmark : ['nano', 'small', 'medium']
Warm-up frames        : 10
Benchmark frames      : 100
Confidence threshold  : 0.25
Setup complete


In [ ]:
# ── Extracting benchmark frames ─────────────────────────────────────────
# I extracted a fixed set of frames from the video to ensure all model variants are benchmarked on identical inputs.
# I resize all frames to 640×480 px — a standard inference resolution so the benchmark is not confounded by different input resolutions.

real_video = PROJECT_ROOT / "data" / "raw" / "sample_traffic.mp4"
mock_video = PROJECT_ROOT / "data" / "mock" / "mock_video.mp4"
VIDEO_PATH = real_video if real_video.exists() else mock_video

BENCH_WIDTH = cfg["benchmark"]["frame_width"]
BENCH_HEIGHT = cfg["benchmark"]["frame_height"]
TOTAL_FRAMES_NEEDED = N_WARMUP + N_BENCH

print(f"Extracting {TOTAL_FRAMES_NEEDED} frames from: {VIDEO_PATH.name}")
print(f"Resizing to: {BENCH_WIDTH}×{BENCH_HEIGHT} px")

import cv2
raw_frames = frames_from_video(VIDEO_PATH, max_frames=TOTAL_FRAMES_NEEDED)

# Resize all frames to the benchmark resolution
bench_frames = [
    cv2.resize(f, (BENCH_WIDTH, BENCH_HEIGHT)) for f in raw_frames
]

# Pad with copies if the video is shorter than needed
while len(bench_frames) < TOTAL_FRAMES_NEEDED:
    bench_frames.extend(bench_frames[:TOTAL_FRAMES_NEEDED - len(bench_frames)])

print(f"\n✓ Benchmark frames ready: {len(bench_frames)} frames at {BENCH_WIDTH}×{BENCH_HEIGHT} px")

Extracting 110 frames from: sample_traffic.mp4
Resizing to: 640×480 px
Extracted 110 frames from sample_traffic.mp4

✓ Benchmark frames ready: 110 frames at 640×480 px


In [ ]:
# ── Running benchmark for each model variant ──────────────────────────────

MODELS_DIR = PROJECT_ROOT / "models"
benchmark_results = []

variant_labels = {
    "nano":   "YOLOv8-nano",
    "small":  "YOLOv8-small",
    "medium": "YOLOv8-medium",
}
weight_files = {
    "nano":   "yolov8n.pt",
    "small":  "yolov8s.pt",
    "medium": "yolov8m.pt",
}

for variant in VARIANTS:
    print(f"\n{'='*50}")
    model = load_model(variant, models_dir=MODELS_DIR)
    weight_path = MODELS_DIR / weight_files[variant]
    
    result = benchmark_model(
        model=model,
        frames=bench_frames,
        model_name=variant_labels[variant],
        weight_path=weight_path,
        confidence=CONF,
        n_warmup=N_WARMUP,
        verbose=True,
    )
    benchmark_results.append(result)
    
    # Clear model from memory before loading the next one
    del model
    import gc; gc.collect()

print(f"\n\n✓ Benchmarking complete for {len(benchmark_results)} variants")


Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 NANO | Classes: 80
Benchmarking: YOLOv8-nano
  Frames     : 110 (first 10 are warm-up)
  Frame size : 640×480 px
  Confidence : 0.25
  Running 10 warm-up frames...
  Frame 20/100 | Running mean: 88.1 ms (11.4 FPS)
  Frame 40/100 | Running mean: 91.5 ms (10.9 FPS)
  Frame 60/100 | Running mean: 99.3 ms (10.1 FPS)
  Frame 80/100 | Running mean: 102.6 ms (9.7 FPS)
  Frame 100/100 | Running mean: 104.3 ms (9.6 FPS)

  ── Results for YOLOv8-nano ──
  Mean latency   : 104.3 ms
  Mean FPS       : 9.6
  p95 latency    : 124.3 ms
  p99 latency    : 145.7 ms
  Peak mem (Δ)   : 1.3 MB

Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80
Benchmarking: YOLOv8-small
  Frames     : 110 (first 10 are warm-up)
  Frame size : 640×480 px
  Confidence : 0.25
  Running 10 warm-up frames...

In [ ]:
# ── Building comparison DataFrame ───────────────────────────────────────
from detection_system.benchmark import compare_benchmarks

comparison_df = compare_benchmarks(benchmark_results)
print("Model comparison table:")
display(comparison_df[[
    "model_name", "model_size_mb", "mean_fps", "mean_latency_ms",
    "median_latency_ms", "p95_latency_ms", "peak_memory_mb"
]].to_string(index=False))

# Save results
comparison_df.to_csv(PROJECT_ROOT / "data" / "processed" / "benchmark_results.csv", index=False)
save_table(comparison_df, "06_benchmark_comparison")
print("\n✓ Benchmark results saved")

Model comparison table:


'   model_name  model_size_mb  mean_fps  mean_latency_ms  median_latency_ms  p95_latency_ms  peak_memory_mb\n  YOLOv8-nano            NaN      9.58           104.35             106.99          124.30            1.26\n YOLOv8-small            NaN      4.37           229.06             216.33          275.96            4.98\nYOLOv8-medium            NaN      1.20           831.70             653.25         1388.52            0.48'

  Saved: reports\tables\06_benchmark_comparison.csv
  Saved: reports\tables\06_benchmark_comparison.tex
  Saved: paper\tables\06_benchmark_comparison.csv
  Saved: paper\tables\06_benchmark_comparison.tex

✓ Benchmark results saved


In [ ]:
# ── FPS comparison bar chart ─────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = sns.color_palette("colorblind", n_colors=len(comparison_df))

# FPS
bars = axes[0].bar(comparison_df["model_name"], comparison_df["mean_fps"],
                   color=palette, edgecolor="white")
for bar, val in zip(bars, comparison_df["mean_fps"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Frames per second", fontsize=11)
axes[0].set_title("Throughput (FPS)", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", alpha=0.4)

# Latency (mean, p95)
x = np.arange(len(comparison_df))
w = 0.35
axes[1].bar(x - w/2, comparison_df["mean_latency_ms"], width=w,
            color=palette, alpha=0.9, label="Mean")
axes[1].bar(x + w/2, comparison_df["p95_latency_ms"], width=w,
            color=palette, alpha=0.5, label="p95", hatch="//")
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df["model_name"], rotation=10)
axes[1].set_ylabel("Latency (ms/frame)", fontsize=11)
axes[1].set_title("Latency — Mean vs. p95", fontsize=12, fontweight="bold")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.4)

# Model size
bars = axes[2].bar(comparison_df["model_name"], comparison_df["model_size_mb"],
                   color=palette, edgecolor="white")
for bar, val in zip(bars, comparison_df["model_size_mb"]):
    if not np.isnan(val):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f"{val:.0f} MB", ha="center", fontsize=10)
axes[2].set_ylabel("Model size (MB)", fontsize=11)
axes[2].set_title("Model Size on Disk", fontsize=12, fontweight="bold")
axes[2].grid(axis="y", alpha=0.4)

plt.suptitle("YOLOv8 Model Variant Comparison — Engineering Metrics",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_figure(fig, "06_fps_comparison")
plt.show()

  Saved: reports\figures\06_fps_comparison.png
  Saved: reports\figures\06_fps_comparison.pdf
  Saved: paper\figures\06_fps_comparison.png
  Saved: paper\figures\06_fps_comparison.pdf


In [ ]:
# ── Latency distribution — violin plots ───────────────────────────────

# Build a long-format DataFrame for seaborn
latency_rows = []
for result in benchmark_results:
    for lat in result.latencies_ms:
        latency_rows.append({"model": result.model_name, "latency_ms": lat})

latency_long = pd.DataFrame(latency_rows)

fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(
    data=latency_long,
    x="model",
    y="latency_ms",
    palette="colorblind",
    ax=ax,
    inner="box",   # Show median + IQR inside violin
)

ax.set_xlabel("Model variant", fontsize=12)
ax.set_ylabel("Latency (ms/frame)", fontsize=12)
ax.set_title(
    f"Per-Frame Latency Distribution — {N_BENCH} Benchmark Frames per Model",
    fontsize=13, fontweight="bold"
)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()

save_figure(fig, "06_latency_distribution")
plt.show()

# Interpretation note
print("\nInterpretation:")
print("  The violin width shows the density of latency values at each speed.")
print("  A narrow, tall violin = consistent latency (predictable throughput).")
print("  A wide violin with long tails = high jitter (unpredictable, bad for real-time).")

C:\Users\Peter\AppData\Local\Temp\ipykernel_25960\3641456639.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(


  Saved: reports\figures\06_latency_distribution.png
  Saved: reports\figures\06_latency_distribution.pdf
  Saved: paper\figures\06_latency_distribution.png
  Saved: paper\figures\06_latency_distribution.pdf

Interpretation:
  The violin width shows the density of latency values at each speed.
  A narrow, tall violin = consistent latency (predictable throughput).
  A wide violin with long tails = high jitter (unpredictable, bad for real-time).


In [ ]:
# ── Step 6: Speed vs. accuracy trade-off summary ──────────────────────────────

print("=" * 60)
print("BENCHMARK SUMMARY — Speed vs. Model Scale Trade-off")
print("=" * 60)

for _, row in comparison_df.iterrows():
    fps = row["mean_fps"]
    lat = row["mean_latency_ms"]
    size = row["model_size_mb"]
    mem = row["peak_memory_mb"]
    print(f"\n{row['model_name']}:")
    print(f"  FPS          : {fps:.1f} frames/sec")
    print(f"  Latency      : {lat:.1f} ms/frame (p95: {row['p95_latency_ms']:.1f} ms)")
    print(f"  Model size   : {size:.0f} MB" if not np.isnan(size) else "  Model size   : N/A")
    print(f"  Peak RAM (Δ) : {mem:.0f} MB")

print()
print("Interpretation (cautious):")
print("  These results are consistent with the expected ordering:")
print("  smaller models provide higher throughput at the cost of accuracy.")
print("  Absolute FPS values depend strongly on hardware — CPU type, RAM speed,")
print("  and thermal conditions. Results should not be compared to GPU benchmarks.")
print("  GPU inference would increase FPS by 5–20× depending on hardware.")

BENCHMARK SUMMARY — Speed vs. Model Scale Trade-off

YOLOv8-nano:
  FPS          : 9.6 frames/sec
  Latency      : 104.3 ms/frame (p95: 124.3 ms)
  Model size   : N/A
  Peak RAM (Δ) : 1 MB

YOLOv8-small:
  FPS          : 4.4 frames/sec
  Latency      : 229.1 ms/frame (p95: 276.0 ms)
  Model size   : N/A
  Peak RAM (Δ) : 5 MB

YOLOv8-medium:
  FPS          : 1.2 frames/sec
  Latency      : 831.7 ms/frame (p95: 1388.5 ms)
  Model size   : N/A
  Peak RAM (Δ) : 0 MB

Interpretation (cautious):
  These results are consistent with the expected ordering:
  smaller models provide higher throughput at the cost of accuracy.
  Absolute FPS values depend strongly on hardware — CPU type, RAM speed,
  and thermal conditions. Results should not be compared to GPU benchmarks.
  GPU inference would increase FPS by 5–20× depending on hardware.
